# MGCI Model Training Pipeline

**SDG Indicator 15.4.2: Mountain Green Cover Index**

This notebook trains a U-Net model for vegetation segmentation using the data exported from Notebook 1.

### Pipeline Overview
1. Load and verify GeoTIFF patches
2. Create spatial train/validation/test split
3. Build U-Net model with ResNet-50 encoder
4. Train with Focal + Dice loss
5. Evaluate and visualize results
6. Save model for inference

**Input:** Patches from Notebook 1  
**Output:** Trained model + evaluation metrics

## 1. Setup and Configuration

In [ ]:
!pip install -q rasterio segmentation-models-pytorch albumentations
print("Dependencies installed")

In [ ]:
import os
import glob
import json
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from tqdm.auto import tqdm

import rasterio
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from sklearn.metrics import confusion_matrix, classification_report

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Libraries imported")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted")

In [ ]:
class Config:
    """Training configuration parameters."""
    
    # Data paths - UPDATE FOR YOUR SETUP
    DATA_DIR = '/content/drive/MyDrive/MGCI_Jizan_2024'
    OUTPUT_DIR = '/content/outputs'
    MODEL_PATH = '/content/drive/MyDrive/MGCI_Jizan_2024/model_best.pth'

    # Band configuration (from Notebook 1 export)
    BAND_NAMES = ['Blue', 'Green', 'Red', 'NIR', 'Elevation', 'Slope', 'VegLabel']
    INPUT_BANDS = [0, 1, 2, 3, 4, 5]  # First 6 bands as input
    LABEL_BAND = 6                     # Last band as label
    N_CHANNELS = 6

    # Model architecture
    MODEL = 'Unet'
    ENCODER = 'resnet50'
    ENCODER_WEIGHTS = 'imagenet'

    # Training hyperparameters
    BATCH_SIZE = 8
    EPOCHS = 80
    LR = 3e-4
    WEIGHT_DECAY = 1e-4
    PATIENCE = 15

    # Data split ratios
    TRAIN_RATIO = 0.70
    VAL_RATIO = 0.15
    TEST_RATIO = 0.15

    # Random seed for reproducibility
    SEED = 42


# Create output directory
os.makedirs(Config.OUTPUT_DIR, exist_ok=True)

# Set random seeds for reproducibility
random.seed(Config.SEED)
np.random.seed(Config.SEED)
torch.manual_seed(Config.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(Config.SEED)

print(f"Configuration:")
print(f"  Data: {Config.DATA_DIR}")
print(f"  Model: {Config.MODEL} + {Config.ENCODER}")
print(f"  Epochs: {Config.EPOCHS}")
print(f"  Batch size: {Config.BATCH_SIZE}")
print(f"  Learning rate: {Config.LR}")

## 2. Data Loading and Verification

In [ ]:
# Find all GeoTIFF patches
all_files = sorted(glob.glob(f'{Config.DATA_DIR}/*.tif'))
print(f"Found {len(all_files)} patches")

if len(all_files) == 0:
    raise FileNotFoundError(f"No .tif files found in {Config.DATA_DIR}")

In [ ]:
# Verify data structure with sample patch
print("Verifying data structure...")

with rasterio.open(all_files[0]) as src:
    data = src.read().astype(np.float32)
    data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)

print(f"  Shape: {data.shape}")
print(f"  Bands: {data.shape[0]}")
print(f"  Size: {data.shape[1]}x{data.shape[2]}")

# Verify band values
print(f"\nBand statistics:")
for i, name in enumerate(Config.BAND_NAMES):
    band = data[i]
    print(f"  {name:12s}: min={band.min():8.2f}, max={band.max():8.2f}, mean={band.mean():8.2f}")

# Check label values
label = data[Config.LABEL_BAND]
unique_vals = np.unique(label)
print(f"\nLabel unique values: {unique_vals}")
if set(unique_vals) == {0.0, 1.0} or set(unique_vals) == {0.0} or set(unique_vals) == {1.0}:
    print("Label values verified (binary 0/1)")
else:
    print(f"WARNING: Unexpected label values")

## 3. Spatial Data Split

In [ ]:
def get_coords_from_filename(filepath):
    """Extract coordinates from patch filename for spatial sorting."""
    with rasterio.open(filepath) as src:
        bounds = src.bounds
        return (bounds.bottom, bounds.left)

# Get coordinates for all patches
print("Extracting coordinates for spatial split...")
coords = [(f, get_coords_from_filename(f)) for f in tqdm(all_files, desc="Processing")]

# Sort by X coordinate (longitude) for spatial split
coords_sorted = sorted(coords, key=lambda x: x[1][1])
sorted_files = [c[0] for c in coords_sorted]

# Split data
n_total = len(sorted_files)
n_train = int(n_total * Config.TRAIN_RATIO)
n_val = int(n_total * Config.VAL_RATIO)

train_files = sorted_files[:n_train]
val_files = sorted_files[n_train:n_train + n_val]
test_files = sorted_files[n_train + n_val:]

print(f"\nSpatial split (sorted by longitude):")
print(f"  Train: {len(train_files)} patches ({len(train_files)/n_total*100:.1f}%)")
print(f"  Val:   {len(val_files)} patches ({len(val_files)/n_total*100:.1f}%)")
print(f"  Test:  {len(test_files)} patches ({len(test_files)/n_total*100:.1f}%)")

In [ ]:
# Visualize spatial split
fig, ax = plt.subplots(figsize=(12, 8))

for files, label, color in [
    (train_files, 'Train', '#2ecc71'),
    (val_files, 'Validation', '#f39c12'),
    (test_files, 'Test', '#e74c3c')
]:
    coords = [get_coords_from_filename(f) for f in files]
    ys = [c[0] for c in coords]
    xs = [c[1] for c in coords]
    ax.scatter(xs, ys, c=color, label=f'{label} ({len(files)})', alpha=0.6, s=20)

ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title('Spatial Train/Validation/Test Split', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{Config.OUTPUT_DIR}/spatial_split.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {Config.OUTPUT_DIR}/spatial_split.png")

In [ ]:
# Calculate normalization statistics from training data only
print("Calculating normalization statistics from training data...")

n_sample = min(200, len(train_files))
band_values = {i: [] for i in Config.INPUT_BANDS}

for f in tqdm(random.sample(train_files, n_sample), desc="Sampling"):
    with rasterio.open(f) as src:
        data = src.read().astype(np.float32)
    data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)
    
    for i in Config.INPUT_BANDS:
        band_values[i].extend(data[i].flatten()[::50])

MEANS = [np.mean(band_values[i]) for i in Config.INPUT_BANDS]
STDS = [max(np.std(band_values[i]), 1.0) for i in Config.INPUT_BANDS]

print(f"\n{'Band':<12} {'Mean':>12} {'Std':>12}")
print("-" * 40)
for i, idx in enumerate(Config.INPUT_BANDS):
    print(f"{Config.BAND_NAMES[idx]:<12} {MEANS[i]:>12.2f} {STDS[i]:>12.2f}")
print("-" * 40)
print("Normalization statistics calculated")

## 4. Dataset and DataLoaders

In [ ]:
class MGCIDataset(Dataset):
    """Dataset for MGCI vegetation segmentation."""

    def __init__(self, files, means, stds, augment=False):
        self.files = files
        self.means = np.array(means, dtype=np.float32).reshape(-1, 1, 1)
        self.stds = np.array(stds, dtype=np.float32).reshape(-1, 1, 1)
        self.augment = augment

        if augment:
            self.transform = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.ShiftScaleRotate(
                    shift_limit=0.1,
                    scale_limit=0.15,
                    rotate_limit=45,
                    p=0.5,
                    border_mode=0
                ),
                A.OneOf([
                    A.GaussNoise(var_limit=(10, 50), p=1),
                    A.GaussianBlur(blur_limit=(3, 5), p=1),
                ], p=0.3),
                A.RandomBrightnessContrast(
                    brightness_limit=0.2,
                    contrast_limit=0.2,
                    p=0.3
                ),
            ])
        else:
            self.transform = None

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        with rasterio.open(self.files[idx]) as src:
            data = src.read().astype(np.float32)
        
        data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)
        
        img = data[Config.INPUT_BANDS].copy()
        label = data[Config.LABEL_BAND].copy()
        
        # Normalize
        img = (img - self.means) / self.stds
        img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)
        
        # Apply augmentation
        if self.transform:
            img_hwc = img.transpose(1, 2, 0)
            transformed = self.transform(image=img_hwc, mask=label)
            img = transformed['image'].transpose(2, 0, 1)
            label = transformed['mask']
        
        img = torch.from_numpy(img.copy()).float()
        label = torch.from_numpy(label.copy()).float().unsqueeze(0)
        
        img = torch.nan_to_num(img, nan=0.0)
        label = torch.nan_to_num(label, nan=0.0)
        
        return img, label


print("Dataset class defined")

In [ ]:
# Create datasets
train_dataset = MGCIDataset(train_files, MEANS, STDS, augment=True)
val_dataset = MGCIDataset(val_files, MEANS, STDS, augment=False)
test_dataset = MGCIDataset(test_files, MEANS, STDS, augment=False)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"DataLoaders created")
print(f"  Train: {len(train_loader)} batches")
print(f"  Val:   {len(val_loader)} batches")
print(f"  Test:  {len(test_loader)} batches")

In [ ]:
# Verify DataLoader output
print("Verifying DataLoader output...")

img_batch, label_batch = next(iter(train_loader))

print(f"  Image shape: {img_batch.shape} (expected: [B, 6, H, W])")
print(f"  Label shape: {label_batch.shape} (expected: [B, 1, H, W])")
print(f"  Image range: [{img_batch.min().item():.4f}, {img_batch.max().item():.4f}]")
print(f"  Label unique: {torch.unique(label_batch).tolist()}")
print(f"  Has NaN: {torch.isnan(img_batch).any().item() or torch.isnan(label_batch).any().item()}")

if not torch.isnan(img_batch).any() and img_batch.shape[1] == 6:
    print("DataLoader verified - ready for training")

## 5. Model Definition

In [ ]:
# Create U-Net model with pretrained ResNet-50 encoder
model = smp.Unet(
    encoder_name=Config.ENCODER,
    encoder_weights=Config.ENCODER_WEIGHTS,
    in_channels=Config.N_CHANNELS,
    classes=1,
    activation=None
)

model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model: {Config.MODEL} + {Config.ENCODER}")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Input channels: {Config.N_CHANNELS}")
print(f"  Output classes: 1 (binary segmentation)")

In [ ]:
class FocalDiceLoss(nn.Module):
    """Combined Focal Loss and Dice Loss for handling class imbalance."""

    def __init__(self, alpha=0.25, gamma=2.0, dice_weight=0.5, focal_weight=0.5):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.dice_weight = dice_weight
        self.focal_weight = focal_weight

    def focal_loss(self, pred, target):
        pred_sigmoid = torch.sigmoid(pred)
        bce = nn.functional.binary_cross_entropy_with_logits(pred, target, reduction='none')
        pt = torch.where(target == 1, pred_sigmoid, 1 - pred_sigmoid)
        focal_weight = self.alpha * (1 - pt) ** self.gamma
        return (focal_weight * bce).mean()

    def dice_loss(self, pred, target, smooth=1.0):
        pred = torch.sigmoid(pred)
        pred_flat = pred.view(-1)
        target_flat = target.view(-1)
        intersection = (pred_flat * target_flat).sum()
        union = pred_flat.sum() + target_flat.sum()
        dice = (2. * intersection + smooth) / (union + smooth)
        return 1 - dice

    def forward(self, pred, target):
        focal = self.focal_loss(pred, target)
        dice = self.dice_loss(pred, target)
        return self.focal_weight * focal + self.dice_weight * dice


# Initialize loss, optimizer, and scheduler
criterion = FocalDiceLoss(alpha=0.25, gamma=2.0, dice_weight=0.5, focal_weight=0.5)
optimizer = optim.AdamW(model.parameters(), lr=Config.LR, weight_decay=Config.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=10,
    T_mult=2,
    eta_min=1e-6
)

print("Training components:")
print(f"  Loss: Focal + Dice (0.5 + 0.5)")
print(f"  Optimizer: AdamW (lr={Config.LR}, wd={Config.WEIGHT_DECAY})")
print(f"  Scheduler: CosineAnnealingWarmRestarts (T_0=10)")

## 6. Training Loop

In [ ]:
def calculate_metrics(pred, target, threshold=0.5):
    """Calculate IoU and Dice metrics."""
    pred_binary = (torch.sigmoid(pred) > threshold).float()
    
    intersection = (pred_binary * target).sum()
    union = pred_binary.sum() + target.sum() - intersection
    
    iou = (intersection + 1e-6) / (union + 1e-6)
    dice = (2 * intersection + 1e-6) / (pred_binary.sum() + target.sum() + 1e-6)
    
    return iou.item(), dice.item()


def train_epoch(model, loader, criterion, optimizer):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    total_iou = 0
    total_dice = 0
    
    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        iou, dice = calculate_metrics(outputs, labels)
        total_loss += loss.item()
        total_iou += iou
        total_dice += dice
    
    n_batches = len(loader)
    return total_loss/n_batches, total_iou/n_batches, total_dice/n_batches


def validate_epoch(model, loader, criterion):
    """Validate for one epoch."""
    model.eval()
    total_loss = 0
    total_iou = 0
    total_dice = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            iou, dice = calculate_metrics(outputs, labels)
            total_loss += loss.item()
            total_iou += iou
            total_dice += dice
    
    n_batches = len(loader)
    return total_loss/n_batches, total_iou/n_batches, total_dice/n_batches


print("Training functions defined")

In [ ]:
# Training loop
print("=" * 90)
print(f"{'TRAINING U-NET':^90}")
print("=" * 90)
print(f"\n{'Ep':>4} {'TrLoss':>10} {'VaLoss':>10} {'TrIoU':>10} {'VaIoU':>10} {'TrDice':>10} {'VaDice':>10} {'LR':>12}")
print("-" * 90)

history = {
    'train_loss': [], 'val_loss': [],
    'train_iou': [], 'val_iou': [],
    'train_dice': [], 'val_dice': [],
    'lr': []
}

best_val_loss = float('inf')
best_val_iou = 0
best_epoch = 0
patience_counter = 0

for epoch in range(Config.EPOCHS):
    # Train
    train_loss, train_iou, train_dice = train_epoch(model, train_loader, criterion, optimizer)
    
    # Validate
    val_loss, val_iou, val_dice = validate_epoch(model, val_loader, criterion)
    
    # Update scheduler
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    
    # Record history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_iou'].append(train_iou)
    history['val_iou'].append(val_iou)
    history['train_dice'].append(train_dice)
    history['val_dice'].append(val_dice)
    history['lr'].append(current_lr)
    
    # Check for improvement
    improved = ''
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_iou = val_iou
        best_epoch = epoch + 1
        patience_counter = 0
        improved = '*'
        
        # Save best model
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_iou': val_iou,
            'means': MEANS,
            'stds': STDS
        }, Config.MODEL_PATH)
    else:
        patience_counter += 1
    
    print(f"{epoch+1:>4} {train_loss:>10.4f} {val_loss:>10.4f} {train_iou:>10.4f} {val_iou:>10.4f} {train_dice:>10.4f} {val_dice:>10.4f} {current_lr:>12.6f} {improved}")
    
    # Early stopping
    if patience_counter >= Config.PATIENCE:
        print(f"\nEarly stopping at epoch {epoch + 1}")
        break

print("-" * 90)
print(f"\nTraining complete")
print(f"  Best epoch: {best_epoch}")
print(f"  Best val loss: {best_val_loss:.4f}")
print(f"  Best val IoU: {best_val_iou:.4f}")
print(f"  Model saved: {Config.MODEL_PATH}")

## 7. Evaluation and Visualization

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss
axes[0, 0].plot(history['train_loss'], label='Train', color='blue')
axes[0, 0].plot(history['val_loss'], label='Validation', color='orange')
axes[0, 0].axvline(best_epoch-1, color='green', linestyle='--', label=f'Best ({best_epoch})')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training and Validation Loss', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# IoU
axes[0, 1].plot(history['train_iou'], label='Train', color='blue')
axes[0, 1].plot(history['val_iou'], label='Validation', color='orange')
axes[0, 1].axvline(best_epoch-1, color='green', linestyle='--', label=f'Best ({best_epoch})')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('IoU')
axes[0, 1].set_title('Training and Validation IoU', fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Dice
axes[1, 0].plot(history['train_dice'], label='Train', color='blue')
axes[1, 0].plot(history['val_dice'], label='Validation', color='orange')
axes[1, 0].axvline(best_epoch-1, color='green', linestyle='--', label=f'Best ({best_epoch})')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Dice')
axes[1, 0].set_title('Training and Validation Dice', fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Learning rate
axes[1, 1].plot(history['lr'], color='green')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Learning Rate')
axes[1, 1].set_title('Learning Rate Schedule', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{Config.OUTPUT_DIR}/training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {Config.OUTPUT_DIR}/training_history.png")

In [ ]:
# Load best model and evaluate on test set
checkpoint = torch.load(Config.MODEL_PATH)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Loaded best model from epoch {checkpoint['epoch'] + 1}")

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(DEVICE)
        outputs = model(images)
        preds = (torch.sigmoid(outputs) > 0.5).cpu().numpy()
        all_preds.extend(preds.flatten())
        all_labels.extend(labels.numpy().flatten())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Calculate metrics
tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()

accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
dice = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0

print(f"\nTest Set Metrics:")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print(f"  IoU:       {iou:.4f}")
print(f"  Dice:      {dice:.4f}")

In [ ]:
# Plot confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))

cm = confusion_matrix(all_labels, all_preds)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Non-Vegetation', 'Vegetation'],
            yticklabels=['Non-Vegetation', 'Vegetation'])
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Confusion Matrix (Test Set)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{Config.OUTPUT_DIR}/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {Config.OUTPUT_DIR}/confusion_matrix.png")

In [ ]:
# Plot sample predictions
fig, axes = plt.subplots(4, 4, figsize=(16, 16))

sample_indices = random.sample(range(len(test_dataset)), 4)

for i, idx in enumerate(sample_indices):
    img, label = test_dataset[idx]
    
    with torch.no_grad():
        pred = model(img.unsqueeze(0).to(DEVICE))
        pred = torch.sigmoid(pred).cpu().squeeze().numpy()
    
    img_np = img.numpy()
    label_np = label.squeeze().numpy()
    
    # RGB composite
    rgb = np.stack([
        img_np[2] * STDS[2] + MEANS[2],
        img_np[1] * STDS[1] + MEANS[1],
        img_np[0] * STDS[0] + MEANS[0]
    ], axis=-1)
    rgb = np.clip(rgb / 3000, 0, 1)
    
    axes[i, 0].imshow(rgb)
    axes[i, 0].set_title('RGB' if i == 0 else '')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(label_np, cmap='Greens', vmin=0, vmax=1)
    axes[i, 1].set_title('Ground Truth' if i == 0 else '')
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(pred, cmap='Greens', vmin=0, vmax=1)
    axes[i, 2].set_title('Prediction' if i == 0 else '')
    axes[i, 2].axis('off')
    
    # Difference
    diff = np.abs(label_np - (pred > 0.5).astype(float))
    axes[i, 3].imshow(diff, cmap='Reds', vmin=0, vmax=1)
    axes[i, 3].set_title('Error' if i == 0 else '')
    axes[i, 3].axis('off')

plt.suptitle('Sample Predictions on Test Set', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{Config.OUTPUT_DIR}/sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {Config.OUTPUT_DIR}/sample_predictions.png")

In [ ]:
# Plot metrics summary
fig, ax = plt.subplots(figsize=(10, 6))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'IoU', 'Dice']
values = [accuracy, precision, recall, f1, iou, dice]

bars = ax.bar(metrics, values, color=['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12', '#1abc9c'])
ax.set_ylim(0, 1)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Test Set Performance Metrics', fontsize=14, fontweight='bold')
ax.axhline(y=0.9, color='gray', linestyle='--', alpha=0.5, label='0.9 threshold')

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{Config.OUTPUT_DIR}/metrics_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {Config.OUTPUT_DIR}/metrics_summary.png")

## 8. Save Training Summary

In [ ]:
# Save training summary to JSON
summary = {
    'timestamp': datetime.now().isoformat(),
    'config': {
        'data_dir': Config.DATA_DIR,
        'model': Config.MODEL,
        'encoder': Config.ENCODER,
        'epochs': Config.EPOCHS,
        'batch_size': Config.BATCH_SIZE,
        'learning_rate': Config.LR,
        'input_channels': Config.N_CHANNELS
    },
    'data_split': {
        'train': len(train_files),
        'val': len(val_files),
        'test': len(test_files)
    },
    'normalization': {
        'means': [float(m) for m in MEANS],
        'stds': [float(s) for s in STDS]
    },
    'training': {
        'best_epoch': best_epoch,
        'best_val_loss': float(best_val_loss),
        'best_val_iou': float(best_val_iou),
        'total_epochs': len(history['train_loss'])
    },
    'test_metrics': {
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
        'iou': float(iou),
        'dice': float(dice)
    },
    'model_path': Config.MODEL_PATH
}

with open(f'{Config.OUTPUT_DIR}/training_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

with open(f'{Config.DATA_DIR}/training_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("Training summary saved")
print(f"  Local: {Config.OUTPUT_DIR}/training_summary.json")
print(f"  Drive: {Config.DATA_DIR}/training_summary.json")

In [ ]:
# Final summary
print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"\nModel Performance (Test Set):")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print(f"  IoU:       {iou:.4f}")
print(f"  Dice:      {dice:.4f}")
print(f"\nOutput Files:")
print(f"  Model: {os.path.basename(Config.MODEL_PATH)}")
print(f"  Figures: training_history.png, confusion_matrix.png,")
print(f"           metrics_summary.png, sample_predictions.png")
print(f"  Summary: training_summary.json")
print(f"\nNext Step: Run Notebook 3 for inference and MGCI calculation")
print("=" * 60)